In [0]:
%run ./00_mounts

In [0]:
from pyspark.sql.functions import current_date, lit, col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

base = f"{SRC}/snowflake"
y = max(f.name.strip("/") for f in dbutils.fs.ls(base))
m = max(f.name.strip("/") for f in dbutils.fs.ls(f"{base}/{y}"))
d = max(f.name.strip("/") for f in dbutils.fs.ls(f"{base}/{y}/{m}"))
latest_path = f"{base}/{y}/{m}/{d}/*.parquet"
file_date = f"{y}-{m}-{d}"

new_df = (spark.read.parquet(latest_path)
          .withColumn("ingest_date", current_date())
          .withColumn("file_date", lit(file_date)))
new_df.printSchema()   # confirm ORDER_ID / ORDER_ITEM_ID case; adjust below if lowercase

w = Window.partitionBy("ORDER_ID","ORDER_ITEM_ID").orderBy(col("ingest_date").desc())
new_df = new_df.withColumn("rn", row_number().over(w)).filter("rn=1").drop("rn")

delta_path = f"{STD}/cust_order_delta"
if not DeltaTable.isDeltaTable(spark, delta_path):
    new_df.write.format("delta").mode("overwrite").option("mergeSchema","true").save(delta_path)
else:
    (DeltaTable.forPath(spark, delta_path).alias("t")
     .merge(new_df.alias("s"), "t.ORDER_ID=s.ORDER_ID AND t.ORDER_ITEM_ID=s.ORDER_ITEM_ID")
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

spark.sql(f"CREATE TABLE IF NOT EXISTS cust_order_delta USING DELTA LOCATION '{delta_path}'")

In [0]:
%sql
SELECT COUNT(*) FROM delta.`abfss://input@bhumeetadlsdevstd01.dfs.core.windows.net/cust_order_delta`;

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-6064584710929818>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT COUNT(*) FROM delta.`abfss://input@bhumeetadlsdevstd01.dfs.core.windows.net/cust_order_delta`;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:183, in SqlMagic.sql(self, line, cell)
    177 exc